In [1]:
import numpy as np
import pandas as pd

# Load from Preprocessing output
df2 = pd.read_csv('data_preprocessed.csv')
df2['capturedAt'] = pd.to_datetime(df2['capturedAt'])
df2['date'] = df2['capturedAt'].dt.date
print(f'Loaded: {df2.shape}')

Loaded: (306226, 29)


# Feature Engineering
Creating new features from existing data.

Features to be created:
1. Discount feature: how deep is the product's discount?
2. Promotion feature: is there an ongoing promotion?
3. Log price: target transformation to make the distribution more normal
4. Historical feature: product prices on previous days

## 1. Discount and Promotion Features
From the raw_discount and promotionId columns, we create new features that are more meaningful for the model.

In [2]:
# How deep is the discount? (0.0 - 1.0)
# raw_discount is already in percent (38 = 38%), divided by 100 to become a decimal
df2['discount_depth'] = df2['raw_discount'] / 100

# Is there an ongoing promotion?
df2['has_promotion'] = (df2['promotionId'] > 0).astype(int)

# Is there a discount?
df2['has_discount'] = (df2['priceBeforeDiscount'] > 0).astype(int)

print(df2[['price', 'priceBeforeDiscount', 'raw_discount',
           'discount_depth', 'has_promotion', 'has_discount']].head(10))

      price  priceBeforeDiscount  raw_discount  discount_depth  has_promotion  \
0  41900000             68000000            38            0.38              1   
1  16900000                    0            14            0.14              0   
2  11900000             13900000            14            0.14              1   
3  16900000                    0            14            0.14              0   
4  16900000                    0            14            0.14              0   
5  16900000                    0            14            0.14              0   
6  11900000             13900000            14            0.14              1   
7  14900000             16900000            14            0.14              1   
8  11900000             13900000            14            0.14              1   
9  16900000                    0            14            0.14              0   

   has_discount  
0             1  
1             0  
2             1  
3             0  
4             0  


- discount_depth 0.38 means a 38% discount from the original price
- has_promotion and has_discount are binary signals; products on promotion tend to have lower prices than usual

## 2. Log Price — Target Transformation
The original price is highly skewed (mean >> median). Models perform better when the target distribution is close to normal. The solution: log(price).

In [3]:
# log1p = log(1 + x), safe for 0 values
df2['log_price'] = np.log1p(df2['price'])

print("Statistical comparison of price vs log_price:")
print(f"\nOriginal price:")
print(f"  Mean   : Rp {df2['price'].mean():,.0f}")
print(f"  Median : Rp {df2['price'].median():,.0f}")
print(f"  Std    : Rp {df2['price'].std():,.0f}")

print(f"\nLog price:")
print(f"  Mean   : {df2['log_price'].mean():.2f}")
print(f"  Median : {df2['log_price'].median():.2f}")
print(f"  Std    : {df2['log_price'].std():.2f}")

Statistical comparison of price vs log_price:

Original price:
  Mean   : Rp 52,341,462
  Median : Rp 20,500,000
  Std    : Rp 91,371,957

Log price:
  Mean   : 16.97
  Median : 16.84
  Std    : 1.27


After the log transformation, the standard deviation is much smaller relative to the mean, making the distribution more balanced. We will use log_price as the target during model training, then convert it back to IDR during prediction using np.expm1() (the inverse of np.log1p()).

## 3. Time-Based Split
Split data based on time, not randomly.Everything before March 22 = train, March 22 = validation.

Why time-based? Because we are predicting the future. The model should only know what happened before the prediction day. If we use a random split, the model could "leak" by learning from the future to predict the past. That is unrealistic.

In [4]:
df2['date'] = df2['capturedAt'].dt.date

train = df2[df2['date'] < pd.Timestamp('2025-03-22').date()].copy()
val   = df2[df2['date'] == pd.Timestamp('2025-03-22').date()].copy()

print(f"Train : {len(train):,} rows  ({train['date'].min()} → {train['date'].max()})")
print(f"Val   : {len(val):,} rows  ({val['date'].min()} → {val['date'].max()})")

Train : 301,355 rows  (2025-01-01 → 2025-03-21)
Val   : 4,871 rows  (2025-03-22 → 2025-03-22)


## 4. Historical Features per Product
This is the most important feature in this project. For each modelId,
we calculate price statistics from historical data:
- model_last_price   → last known price
- model_median_price → median price throughout history
- model_mean_price   → average price throughout history
- model_count        → how many times this product appears in the data

Why is this the most important feature? Because product prices are very stable
from day to day. Yesterday's price is the best predictor
for today's price.

IMPORTANT: historical features are ONLY calculated from the train set, not from val.
Val is "future" data that must not be known during training.

In [5]:
# Calculate from the entire training data
hist = (
    train.groupby('modelId')['price']
    .agg(
        model_last_price='last',
        model_median_price='median',
        model_mean_price='mean',
        model_count='count'
    )
    .reset_index()
)

print(f"Number of unique products: {len(hist):,}")
print(hist.head(10))

Number of unique products: 6,284
    modelId  model_last_price  model_median_price  model_mean_price  \
0  52465307          25500000          25500000.0        25500000.0   
1  52465308          25500000          25500000.0        25500000.0   
2  52465309          25500000          25500000.0        25500000.0   
3  52465310          25500000          25500000.0        25500000.0   
4  56064071          56500000          55500000.0        55900000.0   
5  56199952         139000000         139000000.0       139000000.0   
6  56203109          89000000          89000000.0        89000000.0   
7  59560150          48800000          48800000.0        48800000.0   
8  59561910          35900000          35900000.0        35900000.0   
9  73913241          13500000          13500000.0        13500000.0   

   model_count  
0           35  
1           35  
2           35  
3           35  
4           30  
5           24  
6           61  
7           42  
8           14  
9            2 

Each modelId has its own historical price statistics. model_count tells the model how reliable the prediction for this product is. Products with a lot of history are more reliable than products that only appear 1-2 times.

In [6]:
global_median = train['price'].median()

# Merge into train and val
hist_cols = ['model_last_price', 'model_median_price',
             'model_mean_price', 'model_count']

# Drop first if they already exist (to ensure it's safe to rerun multiple times)
train = train.drop(columns=[c for c in hist_cols if c in train.columns])
val   = val.drop(columns=[c for c in hist_cols if c in val.columns])

train = train.merge(hist, on='modelId', how='left')
val   = val.merge(hist, on='modelId', how='left')

for col in hist_cols:
    train[col] = train[col].fillna(global_median if col != 'model_count' else 0)
    val[col]   = val[col].fillna(global_median if col != 'model_count' else 0)

print(f"Products in val not present in train: {(val['model_count']==0).sum()}")
print(val[['modelId','price','model_last_price','model_median_price']].head())

Products in val not present in train: 8
        modelId     price  model_last_price  model_median_price
0  246950643806  26800000        26800000.0          26800000.0
1  158705684479  15000000        15000000.0          15000000.0
2  158705684478  15000000        15000000.0          15000000.0
3   99039190766  19000000        19900000.0          19900000.0
4   99039190765  19000000        19900000.0          19900000.0


Only 8 products in the validation set are not present in the training set; cold start is very rare in this data. For those 8 products, we fill them with global_median as a fallback.

## 5. Summary of Features to be Used by the Model
Complete list of all features to be fed into the model.

In [7]:
FEATURE_COLS = [
    # Time
    'hour', 'dayofweek', 'month', 'day',
    # Product identity
    'shopId', 'cat_id',
    # Discount & promotion
    'has_promotion', 'has_discount', 'discount_depth', 'show_discount',
    # Reference price
    'item_price_min', 'item_price_max', 'priceBeforeDiscount',
    # Shop signals
    'shop_rating', 'shop_response_rate', 'shop_follower_count',
    'is_official_shop', 'is_verified',
    # Product signals
    'review_rating', 'total_rating_count', 'cmt_count',
    'is_free_shipping', 'is_pre_order',
    # Historical
    'model_last_price', 'model_median_price',
    'model_mean_price', 'model_count'
]

print(f"Total features: {len(FEATURE_COLS)}")
print(f"\nFeature list:")
for i, col in enumerate(FEATURE_COLS, 1):
    print(f"  {i:2}. {col}")


Total features: 27

Feature list:
   1. hour
   2. dayofweek
   3. month
   4. day
   5. shopId
   6. cat_id
   7. has_promotion
   8. has_discount
   9. discount_depth
  10. show_discount
  11. item_price_min
  12. item_price_max
  13. priceBeforeDiscount
  14. shop_rating
  15. shop_response_rate
  16. shop_follower_count
  17. is_official_shop
  18. is_verified
  19. review_rating
  20. total_rating_count
  21. cmt_count
  22. is_free_shipping
  23. is_pre_order
  24. model_last_price
  25. model_median_price
  26. model_mean_price
  27. model_count


In [8]:
import pickle

# Save train, val, FEATURE_COLS, and global_median for next notebook
train.to_csv('data_train.csv', index=False)
val.to_csv('data_val.csv', index=False)
with open('feature_cols.pkl', 'wb') as f:
    pickle.dump({'FEATURE_COLS': FEATURE_COLS, 'global_median': global_median}, f)
print('✓ Saved: data_train.csv, data_val.csv, feature_cols.pkl')

✓ Saved: data_train.csv, data_val.csv, feature_cols.pkl
